# 01 · Universo de tickers (SEC EDGAR)

**Objetivo:** obtener la lista de empresas que cotizan en EEUU a partir de la SEC y quedarnos solo con las que cotizan en **NASDAQ y NYSE**, que es el alcance del proyecto. Antes de usar los datos se inspecciona la respuesta cruda de la SEC, para no asumir su formato.

**Entradas:**
- `https://www.sec.gov/files/company_tickers_exchange.json` (cacheado en `data/cache/company_tickers_exchange.parquet` y `data/cache/company_tickers_exchange_crudo.json`)
- `https://www.sec.gov/files/company_tickers.json` (solo para comparar; cacheado en `data/cache/company_tickers_crudo.json`)
- Variable `SEC_EDGAR_USER_AGENT` del `.env`

**Salidas:**
- `data/cache/universo_nasdaq_nyse.parquet` — tickers de NASDAQ + NYSE con su CIK (entrada del notebook 02)

In [1]:
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

from funciones import (
    CARPETA_CACHE,
    URL_TICKERS_CON_BOLSA,
    cargar_universo_tickers,
    descargar_json_sec,
    filtrar_por_bolsa,
    obtener_cik,
)

load_dotenv();

## 1. JSON crudo de la SEC (antes de parsear)

Primero miramos exactamente qué envía la SEC, antes de convertirlo en tabla. La respuesta se guarda en `data/cache/` para que volver a ejecutar el notebook no repita la descarga.

**Nota de honestidad metodológica:** el fichero documentado oficialmente por la SEC (en su FAQ, pregunta *"Do you have a file that maps company ticker / CIK / company name?"*) es `company_tickers.json`. La variante `company_tickers_exchange.json` que usamos **no** aparece en esa FAQ ni en la documentación de las APIs de EDGAR: es un endpoint real (responde `200 OK`), mantenido por la SEC en el mismo directorio, pero no listado oficialmente. Lo usamos porque es el único que trae el campo `exchange`, imprescindible para filtrar a NASDAQ/NYSE.

In [2]:
crudo_con_bolsa = descargar_json_sec(
    URL_TICKERS_CON_BOLSA, ruta_cache=CARPETA_CACHE / "company_tickers_exchange_crudo.json"
)

print("Claves de nivel superior:", list(crudo_con_bolsa.keys()))
print("Campos (fields):", crudo_con_bolsa["fields"])
print("Nº de filas en 'data':", len(crudo_con_bolsa["data"]))
print("Primeras 5 filas crudas:")
for fila in crudo_con_bolsa["data"][:5]:
    print("  ", fila)

Claves de nivel superior: ['fields', 'data']
Campos (fields): ['cik', 'name', 'ticker', 'exchange']
Nº de filas en 'data': 10435
Primeras 5 filas crudas:
   [1045810, 'NVIDIA CORP', 'NVDA', 'Nasdaq']
   [320193, 'Apple Inc.', 'AAPL', 'Nasdaq']
   [1652044, 'Alphabet Inc.', 'GOOGL', 'Nasdaq']
   [789019, 'MICROSOFT CORP', 'MSFT', 'Nasdaq']
   [1018724, 'AMAZON COM INC', 'AMZN', 'Nasdaq']


El formato es **tabular**: una lista `fields` con los nombres de las columnas y una lista `data` con una fila por empresa. Por eso `cargar_universo_tickers()` lo convierte directamente en un DataFrame.

Para comparar, el fichero documentado `company_tickers.json` tiene otra estructura (un diccionario de diccionarios) y **no** trae la bolsa:

In [3]:
crudo_documentado = descargar_json_sec(
    "https://www.sec.gov/files/company_tickers.json",
    ruta_cache=CARPETA_CACHE / "company_tickers_crudo.json",
)

print("Tipo del objeto raíz:", type(crudo_documentado).__name__)
print("Nº de entradas:", len(crudo_documentado))
print("Primeras 3 entradas:")
for clave in list(crudo_documentado)[:3]:
    print("  ", clave, "->", crudo_documentado[clave])

Tipo del objeto raíz: dict
Nº de entradas: 10435
Primeras 3 entradas:
   0 -> {'cik_str': 1045810, 'ticker': 'NVDA', 'title': 'NVIDIA CORP'}
   1 -> {'cik_str': 320193, 'ticker': 'AAPL', 'title': 'Apple Inc.'}
   2 -> {'cik_str': 1652044, 'ticker': 'GOOGL', 'title': 'Alphabet Inc.'}


### ¿Coinciden los dos ficheros?

Si el fichero no documentado fuese poco fiable, esperaríamos que sus CIK no coincidieran con los del documentado. Comparamos los conjuntos de CIK de ambos.

In [4]:
ciks_con_bolsa = {fila[0] for fila in crudo_con_bolsa["data"]}
ciks_documentado = {entrada["cik_str"] for entrada in crudo_documentado.values()}
ciks_comunes = ciks_con_bolsa & ciks_documentado

print("CIK distintos en company_tickers_exchange.json:", len(ciks_con_bolsa))
print("CIK distintos en company_tickers.json:         ", len(ciks_documentado))
print("CIK en ambos:", len(ciks_comunes))
print(f"Coincidencia sobre el fichero documentado: {len(ciks_comunes) / len(ciks_documentado):.1%}")

CIK distintos en company_tickers_exchange.json: 8017
CIK distintos en company_tickers.json:          8017
CIK en ambos: 8017
Coincidencia sobre el fichero documentado: 100.0%


## 2. Universo completo (todas las bolsas)

`cargar_universo_tickers()` devuelve la tabla limpia: el CIK como texto de 10 dígitos (con ceros a la izquierda, que es como lo pide la API de EDGAR) y el ticker en mayúsculas. Se lee de la caché `data/cache/company_tickers_exchange.parquet`, descargada el **16 sep 2026**; por eso su número de filas puede diferir del JSON crudo de arriba, que es una instantánea posterior. Trabajamos con la caché para que los resultados sean reproducibles.

In [5]:
universo_completo = cargar_universo_tickers()

print("Total de entradas (todas las bolsas):", len(universo_completo))
universo_completo["exchange"].value_counts(dropna=False)

Total de entradas (todas las bolsas): 10422


exchange
Nasdaq    4359
NYSE      3300
OTC       2502
None       217
CBOE        44
Name: count, dtype: int64

Además de NASDAQ y NYSE aparecen **OTC** (mercado extrabursátil, poco líquido y con menos requisitos de información), **CBOE** y entradas sin bolsa (`None`: empresas que reportan a la SEC pero no cotizan, p. ej. emisores de deuda privada). Todas quedan fuera del alcance del proyecto.

## 3. Filtro a NASDAQ + NYSE

In [6]:
universo = filtrar_por_bolsa(universo_completo)

print("Tickers NASDAQ + NYSE:", len(universo))
universo.head(10)

Tickers NASDAQ + NYSE: 7659


,cik,ticker,name,exchange
0,0001045810,NVDA,NVIDIA CORP,Nasdaq
1,0000320193,AAPL,Apple Inc.,Nasdaq
2,0001652044,GOOGL,Alphabet Inc.,Nasdaq
3,0000789019,MSFT,MICROSOFT CORP,Nasdaq
4,0001018724,AMZN,AMAZON COM INC,Nasdaq
5,0001730168,AVGO,Broadcom Inc.,Nasdaq
6,0001326801,META,"Meta Platforms, Inc.",Nasdaq
7,0001318605,TSLA,"Tesla, Inc.",Nasdaq
8,0000723125,MU,MICRON TECHNOLOGY INC,Nasdaq
9,0001067983,BRK-B,BERKSHIRE HATHAWAY INC,NYSE


In [7]:
conteo_por_bolsa = universo["exchange"].value_counts()
print(conteo_por_bolsa)

grafico = px.bar(
    x=conteo_por_bolsa.index,
    y=conteo_por_bolsa.values,
    labels={"x": "Bolsa", "y": "Nº de tickers"},
    title="Tickers por bolsa (NASDAQ + NYSE)",
)
# "notebook_connected" carga plotly desde internet en vez de incrustarlo (el .ipynb pesa KB, no MB)
grafico.show(renderer="notebook_connected")

exchange
Nasdaq    4359
NYSE      3300
Name: count, dtype: int64


## 4. Búsqueda ticker → CIK

El CIK es el identificador que usa la SEC; lo necesitaremos en los notebooks 03 y 07 para pedir los filings de cada empresa. `obtener_cik()` lanza un error claro si el ticker no existe, en lugar de devolver un valor vacío en silencio.

In [8]:
for ticker in ["AAPL", "NVDA", "TSLA", "tsla"]:
    print(ticker, "->", obtener_cik(ticker, universo))

try:
    obtener_cik("TICKERFALSO", universo)
except ValueError as error:
    print("Error esperado:", error)

AAPL -> 0000320193
NVDA -> 0001045810
TSLA -> 0001318605
tsla -> 0001318605
Error esperado: El ticker 'TICKERFALSO' no está en el universo de la SEC.


Búsqueda libre por nombre de empresa (sin distinguir mayúsculas, coincidencia parcial). Útil para comprobar a mano que una empresa concreta está en el universo:

In [9]:
texto_buscado = "apple"
universo[universo["name"].str.contains(texto_buscado, case=False, na=False)]

,cik,ticker,name,exchange
1,0000320193,AAPL,Apple Inc.,Nasdaq
1582,0001418121,APLE,"Apple Hospitality REIT, Inc.",NYSE
3559,0000063330,MLP,MAUI LAND & PINEAPPLE CO INC,NYSE
5036,0001938109,PAPL,Pineapple Financial Inc.,NYSE


## 5. Guardar la salida

Guardamos el universo filtrado para que el notebook 02 lo lea de disco sin depender de este.

In [10]:
ruta_salida = CARPETA_CACHE / "universo_nasdaq_nyse.parquet"
universo.to_parquet(ruta_salida, index=False)
print("Guardado:", ruta_salida.relative_to(CARPETA_CACHE.parent.parent), "-", len(universo), "filas")

Guardado: data/cache/universo_nasdaq_nyse.parquet - 7659 filas


## Comprobaciones

Las cifras exactas corresponden a la caché del 16 sep 2026. Si se fuerza una descarga nueva (`forzar_descarga=True`), los recuentos cambiarán y habrá que actualizar estos `assert`.

In [11]:
# Recuentos verificados sobre la caché
assert len(universo_completo) == 10_422
assert len(universo) == 7_659
assert conteo_por_bolsa["Nasdaq"] == 4_359
assert conteo_por_bolsa["NYSE"] == 3_300

# Solo quedan las dos bolsas del proyecto
assert set(universo["exchange"]) == {"Nasdaq", "NYSE"}

# Formato de las columnas
assert universo["cik"].str.fullmatch(r"\d{10}").all(), "Hay CIK que no tienen 10 dígitos"
assert universo["ticker"].notna().all() and (universo["ticker"] == universo["ticker"].str.upper()).all()
assert universo[["cik", "ticker", "name"]].notna().all().all()

# Búsqueda ticker -> CIK
assert obtener_cik("AAPL", universo) == "0000320193"

# La salida se ha guardado y se puede releer igual
assert pd.read_parquet(ruta_salida).equals(universo)

print("Todas las comprobaciones OK")

Todas las comprobaciones OK
